# Lean Statement Diversity Dataset — Perturbation Pipeline
Rule-based transforms — no API key required. Produces `(anchor, variant, transformation_type)` triples.

In [1]:
import re, json
import pandas as pd
from pathlib import Path

OUTPUT_FILE = Path("perturbation_pairs.jsonl")
print("Ready.")

Ready.


In [2]:
df = pd.read_json(
    "hf://datasets/FrenzyMath/mathlib_informal_v4.19.0/data.jsonl",
    lines=True,
)
theorems = df[df["kind"] == "theorem"].copy()
theorems["sig_len"] = theorems["signature"].str.len()
working_set = theorems[
    (theorems["sig_len"] >= 20) & (theorems["sig_len"] <= 400)
].reset_index(drop=True)
print(f"Total theorems : {len(theorems):,}")
print(f"Working set    : {len(working_set):,}  (signature length 20–400 chars)")
working_set[["name", "signature", "sig_len"]].head(3)

/Users/dqxiang/Projects/Math_AI_Lab/DALAB-May-2026-Lean-wiggle-wiggle/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total theorems : 159,399
Working set    : 156,263  (signature length 20–400 chars)


,name,signature,sig_len
0,"[CategoryTheory, Limits, coequalizer, existsUn...",{W : C} (k : Y ⟶ W) (h : f ≫ k = g ≫ k) : ∃! ...,96
1,"[CategoryTheory, Limits, Fork, ι_ofι]",{P : C} (ι : P ⟶ X) (w : ι ≫ f = ι ≫ g) : (Fo...,63
2,"[CategoryTheory, Limits, parallelPair_obj_one]",(f g : X ⟶ Y) : (parallelPair f g).obj one = Y,47


In [3]:
# ── Signature parser ─────────────────────────────────────────────────────────
def split_signature(sig: str):
    """Return (params_str, conclusion_str) by finding the last top-level colon."""
    depth, last_colon = 0, -1
    i = 0
    while i < len(sig):
        c = sig[i]
        if c in "({[":  depth += 1
        elif c in ")}]": depth -= 1
        elif c == ":" and depth == 0:
            nxt = sig[i + 1] if i + 1 < len(sig) else ""
            if nxt not in ("=", ":"):   # skip := and ::
                last_colon = i
        i += 1
    if last_colon == -1:
        return None
    return sig[:last_colon], sig[last_colon + 1 :].strip()


def first_arrow(s: str):
    """Return index of first top-level → (or ->) in s, else -1."""
    depth, i = 0, 0
    while i < len(s):
        c = s[i]
        if c in "({[": depth += 1
        elif c in ")}]": depth -= 1
        elif depth == 0:
            if s[i : i + 1] == "→":
                return i
            if s[i : i + 2] == "->":
                return i
        i += 1
    return -1


# ── Four transforms ──────────────────────────────────────────────────────────
def apply_negation(sig: str):
    parts = split_signature(sig)
    if parts is None:
        return None
    params, concl = parts
    if ' = ' in concl and '→' not in concl and '->' not in concl:
        return params + ": " + concl.replace(' = ', ' ≠ ', 1), "Changed = to ≠"
    if '≠' in concl:
        return params + ": " + concl.replace('≠', '=', 1), "Changed ≠ to ="
    if concl.startswith('¬'):
        inner = concl[1:].strip().lstrip('(').rstrip(')')
        return params + ": " + inner, "Removed ¬ from conclusion"
    return params + ": ¬(" + concl + ")", "Negated conclusion with ¬"


def apply_contrapositive(sig: str):
    parts = split_signature(sig)
    if parts is None:
        return None
    params, concl = parts
    pos = first_arrow(concl)
    if pos == -1:
        return None
    arrow = '→' if concl[pos] == '→' else '->'
    ante = concl[:pos].strip()
    cons = concl[pos + len(arrow):].strip()
    variant_concl = f'¬({cons}) → ¬({ante})'
    return params + ": " + variant_concl, "Contrapositive: ¬Q → ¬P"


def apply_generalization(sig: str):
    # Remove the first [TypeClass ...] constraint
    m = re.search(r'\s*\[([^\]]+)\]', sig)
    if m:
        variant = sig[:m.start()] + sig[m.end():]
        return variant, f"Removed typeclass [{m.group(1).strip()[:40]}]"
    # Remove an explicit hypothesis that isn't a type-variable declaration
    m = re.search(r'\s*\((\w+\s*:\s*[^()]+)\)', sig)
    if m:
        content = m.group(1)
        if 'Type' not in content and 'Sort' not in content:
            variant = sig[:m.start()] + sig[m.end():]
            return variant, f"Removed hypothesis ({content.strip()[:40]})"
    return None


def apply_bound_perturbation(sig: str):
    specs = [
        (r'(≥\s*)(\d+)', +1, '≥ n → ≥ n+1'),
        (r'(≤\s*)(\d+)', -1, '≤ n → ≤ n-1'),
        (r'(>\s*)(\d+)',  +1, '> n → > n+1'),
        (r'(<\s*)(\d+)',  -1, '< n → < n-1'),
    ]
    for pat, delta, desc in specs:
        m = re.search(pat, sig)
        if m:
            new_val = str(max(0, int(m.group(2)) + delta))
            variant = sig[:m.start(2)] + new_val + sig[m.end(2):]
            return variant, f"Bound perturbation: {desc}"
    return None


TRANSFORMS = {
    "negation":           apply_negation,
    "contrapositive":     apply_contrapositive,
    "generalization":     apply_generalization,
    "bound_perturbation": apply_bound_perturbation,
}
print("Transform functions defined.")

Transform functions defined.


In [4]:
# Smoke test on a handful of theorems
smoke = working_set.sample(5, random_state=7)
for _, row in smoke.iterrows():
    sig = row["signature"]
    name = ".".join(row["name"]) if isinstance(row["name"], list) else row["name"]
    print(f"\n[{name}]")
    print(f"  ANCHOR:  {sig.strip()[:100]}")
    for tname, fn in TRANSFORMS.items():
        result = fn(sig)
        if result:
            variant, desc = result
            print(f"  {tname:<20} {variant.strip()[:80]}  ({desc})")
        else:
            print(f"  {tname:<20} (not applicable)")


[MeasureTheory.integrable_congr']
  ANCHOR:  {f : α → β} {g : α → γ} (hf : AEStronglyMeasurable f μ) (hg : AEStronglyMeasurable g μ) (h : ∀ᵐ a ∂μ
  negation             {f : α → β} {g : α → γ} (hf : AEStronglyMeasurable f μ) (hg : AEStronglyMeasurab  (Negated conclusion with ¬)
  contrapositive       (not applicable)
  generalization       {f : α → β} {g : α → γ} (hg : AEStronglyMeasurable g μ) (h : ∀ᵐ a ∂μ, ‖f a‖ = ‖g  (Removed hypothesis (hf : AEStronglyMeasurable f μ))
  bound_perturbation   (not applicable)

[Fin.insertNth_apply_above]
  ANCHOR:  {i j : Fin (n + 1)} (h : i < j) (x : α i) (p : ∀ k, α (i.succAbove k)) :
  i.insertNth x p j = @Eq.r
  negation             {i j : Fin (n + 1)} (h : i < j) (x : α i) (p : ∀ k, α (i.succAbove k)) : i.inser  (Changed = to ≠)
  contrapositive       (not applicable)
  generalization       {i j : Fin (n + 1)} (x : α i) (p : ∀ k, α (i.succAbove k)) :
  i.insertNth x p j  (Removed hypothesis (h : i < j))
  bound_perturbation   (not applicable)



In [5]:
# ── Full pipeline ─────────────────────────────────────────────────────────────
# Set SAMPLE_SIZE = None to run on the full working set (~100 k theorems, a few seconds)
SAMPLE_SIZE = 1000

sample = (
    working_set.sample(n=SAMPLE_SIZE, random_state=42)
    if SAMPLE_SIZE else working_set
)

pairs = []
skipped = 0

for _, row in sample.iterrows():
    sig    = row["signature"]
    name   = ".".join(row["name"]) if isinstance(row["name"], list) else row["name"]
    module = ".".join(row["module_name"]) if isinstance(row["module_name"], list) else row["module_name"]
    info   = row.get("informal_description", "")

    any_hit = False
    for tname, fn in TRANSFORMS.items():
        result = fn(sig)
        if result is None:
            continue
        variant, desc = result
        if variant.strip() == sig.strip():   # no real change
            continue
        pairs.append({
            "anchor_name":        name,
            "module":             module,
            "anchor_signature":   sig,
            "anchor_informal":    info,
            "variant_signature":  variant,
            "transformation_type": tname,
            "change_description": desc,
        })
        any_hit = True
    if not any_hit:
        skipped += 1

pairs_df = pd.DataFrame(pairs)
print(f"Processed {len(sample):,} theorems")
print(f"Generated {len(pairs_df):,} pairs  |  {skipped} theorems yielded no transform")

Processed 1,000 theorems
Generated 1,822 pairs  |  0 theorems yielded no transform


In [6]:
# Stats
print("=== Transformation distribution ===")
print(pairs_df["transformation_type"].value_counts().to_string())
print()

import math
counts = pairs_df["transformation_type"].value_counts(normalize=True)
entropy = -sum(p * math.log2(p) for p in counts if p > 0)
print(f"Transformation entropy : {entropy:.3f} bits  (target ≥ 1.5)")
print(f"Pairs per theorem      : {len(pairs_df) / len(sample):.2f} on average")

=== Transformation distribution ===
transformation_type
negation              1000
generalization         750
bound_perturbation      37
contrapositive          35

Transformation entropy : 1.226 bits  (target ≥ 1.5)
Pairs per theorem      : 1.82 on average


In [7]:
# Preview
for ttype in pairs_df["transformation_type"].unique():
    row = pairs_df[pairs_df["transformation_type"] == ttype].iloc[0]
    print(f"[{ttype}]")
    print(f"  ANCHOR:  {row['anchor_signature'].strip()[:110]}")
    print(f"  VARIANT: {row['variant_signature'].strip()[:110]}")
    print(f"  NOTE:    {row['change_description']}")
    print()

[negation]
  ANCHOR:  {c : ℝ} {x} (h : ‖x‖ ≤ c) : ‖f x‖ ≤ ‖f‖ * c
  VARIANT: {c : ℝ} {x} (h : ‖x‖ ≤ c) : ¬(‖f x‖ ≤ ‖f‖ * c)
  NOTE:    Negated conclusion with ¬

[generalization]
  ANCHOR:  {c : ℝ} {x} (h : ‖x‖ ≤ c) : ‖f x‖ ≤ ‖f‖ * c
  VARIANT: {c : ℝ} {x} : ‖f x‖ ≤ ‖f‖ * c
  NOTE:    Removed hypothesis (h : ‖x‖ ≤ c)

[contrapositive]
  ANCHOR:  : AntivaryOn f g s ↔ ∀ ⦃i⦄, i ∈ s → ∀ ⦃j⦄, j ∈ s → (f j - f i) • (g j - g i) ≤ 0
  VARIANT: : ¬(∀ ⦃j⦄, j ∈ s → (f j - f i) • (g j - g i) ≤ 0) → ¬(AntivaryOn f g s ↔ ∀ ⦃i⦄, i ∈ s)
  NOTE:    Contrapositive: ¬Q → ¬P

[bound_perturbation]
  ANCHOR:  [Subsingleton α] (s : Set α) : s.ncard ≤ 1
  VARIANT: [Subsingleton α] (s : Set α) : s.ncard ≤ 0
  NOTE:    Bound perturbation: ≤ n → ≤ n-1



In [8]:
# Save
pairs_df.to_json(OUTPUT_FILE, orient="records", lines=True, force_ascii=False)
print(f"Saved {len(pairs_df):,} pairs → {OUTPUT_FILE}")

Saved 1,822 pairs → perturbation_pairs.jsonl
